# CUDA graph 根因 —— 从「强关联」做到「因果实证」

## 现在有什么

2026-09-05 定位到：SGLang 解码 CUDA graph 只捕获 `bs=[1, 2, 4, 8]`（日志原文），
TPOT 在 batch 8→10 之间一步跳 4.8 倍后平坦：

| batch | 8 | 10 | 12 | 14 | 16 | 18 | 32 |
|---|---|---|---|---|---|---|---|
| TPOT p50 (ms) | **6.23** | **29.76** | 29.76 | 28.51 | 31.01 | 31.52 | 31.74 |

**但这只是「日志写着 8」和「台阶在 8 之后」两条吻合，属强关联，不是因果实证。**
本轮用三组配置做**干预实验**：改变图捕获范围，看台阶是否跟着动。

## 三组配置与跑前写死的预测

| 组 | 配置 | 若根因成立，应当观察到 |
|---|---|---|
| A | 默认（基线复现） | 台阶仍在 8 → 10 之间 |
| B | `--cuda-graph-backend-decode disabled`（只关解码图，这正是被检验的机制） | **台阶消失**，batch 4/8 的 TPOT 也升到约 30 ms |
| C | `--cuda-graph-max-bs-decode 32`（其余与 A 完全相同，单变量） | **台阶右移**到实际捕获上限之后；若捕获到 32，则 4–32 全程低 TPOT |

**B 是最干净的一条**：它不是移动边界，而是把整个机制关掉。
若 B 里 batch 8 的 TPOT 仍是 6 ms 量级，**根因解释直接被推翻**。

**每组都必须从日志里读回实际捕获的 `bs=[...]`**，不能假设参数生效了 ——
上一轮就吃过「假设日志措辞」的亏。

## 跑前更正一条旧表述

上一轮写的是「捕获范围被显存卡死：`avail mem=1.91 GB`」。**这个归因是错的。**
SGLang v0.5.19 源码 `python/sglang/srt/arg_groups/memory_hook.py` 按 GPU **总显存**分档：
`gpu_mem < 20*1024 MiB` 这一档（源码注释 `# T4, 4080`）直接把解码图 `max_bs` 钉为 8、
`chunked_prefill_size` 钉为 2048，**与运行时可用显存无关**。
日志里的 `avail mem=1.91 GB` 只是捕获前的显存快照，不是限制原因。
所以 C 组不再下调 `--mem-fraction-static`，只改上限这一个变量。

## 顺带补上未完成的实验 3

默认 flashinfer 后端那组上次 420s 没起来。本轮延长到 600s，
**起不来就把失败原因原样打出来**，不重试掩盖。


## 0. 环境

In [1]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout
print(out)
print("必须也是 Tesla T4，才与前几轮可比。")


name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5

必须也是 Tesla T4，才与前几轮可比。


## 1. 装 SGLang（沿用已验证口径）

In [2]:
import importlib.metadata as md_, subprocess, sys, os, shutil

CHECK = ["sglang", "aiohttp", "torchvision"]
TF_PIN = "transformers==5.12.1"

def ver(p):
    try:
        return md_.version(p)
    except Exception:
        return None

def sh(cmd):
    return subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)

missing = [p for p in CHECK if ver(p) is None]
print("缺失:", missing or "无")
if missing:
    if shutil.which("cargo") is None:
        sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
    print("装 sglang[srt] + aiohttp + torchvision + " + TF_PIN + "（约 5-10 分钟）...")
    r = sh([sys.executable, "-m", "pip", "install", "-q",
            "sglang[srt]", "aiohttp", "torchvision", TF_PIN])
    print("退出码:", r.returncode)
    if r.returncode:
        print(r.stdout[-2500:]); print(r.stderr[-2500:])
else:
    print("三个包都在，跳过安装。")

for pkg in ["kernels", "torchaudio"]:
    u = sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg])
    print("  卸 %-12s 退出码 %s" % (pkg, u.returncode))
if ver("transformers") != "5.12.1":
    sh([sys.executable, "-m", "pip", "install", "-q", TF_PIN])
print()
for p in CHECK + ["transformers", "torch"]:
    print("  %-14s %s" % (p, ver(p) or "（未安装）"))


缺失: ['sglang']
装 sglang[srt] + aiohttp + torchvision + transformers==5.12.1（约 5-10 分钟）...
退出码: 0
  卸 kernels      退出码 0
  卸 torchaudio   退出码 0

  sglang         0.5.19
  aiohttp        3.14.3
  torchvision    0.28.0
  transformers   5.12.1
  torch          2.13.0


## 2. 探测参数 —— 先确认存在，别猜

`--cuda-graph-backend-decode`、`--cuda-graph-max-bs-decode` 是本轮的核心参数
（0.5.19 的正式旗标；旧的 `--disable-cuda-graph` / `--cuda-graph-max-bs` 只是兼容别名）。
**任一不存在就停下来把输出发我。**


In [3]:
import subprocess, sys, re

r = subprocess.run([sys.executable, "-m", "sglang.launch_server", "--help"],
                   capture_output=True, text=True)
h = r.stdout + r.stderr
print("help returncode:", r.returncode, "| 长度:", len(h))
print()
need = ["--cuda-graph-backend-decode", "--cuda-graph-max-bs-decode", "--mem-fraction-static"]
ok = True
for flag in need:
    hit = flag in h
    print("  %-26s %s" % (flag, "存在" if hit else "不存在 !!"))
    ok = ok and hit
print()
for g in sorted(set(re.findall(r"--[a-z0-9-]*cuda-graph[a-z0-9-]*", h))):
    print("   ", g)
print()
print("结论:", "可以往下跑。" if ok else "有参数对不上，先停。")


help returncode: 0 | 长度: 165929

  --cuda-graph-backend-decode 存在
  --cuda-graph-max-bs-decode 存在
  --mem-fraction-static      存在

    --cuda-graph-
    --cuda-graph-backend-
    --cuda-graph-backend-decode
    --cuda-graph-backend-prefill
    --cuda-graph-bs
    --cuda-graph-bs-decode
    --cuda-graph-bs-prefill
    --cuda-graph-config
    --cuda-graph-max-bs
    --cuda-graph-max-bs-decode
    --cuda-graph-max-bs-prefill
    --cuda-graph-tc-compiler
    --debug-cuda-graph
    --disable-cuda-graph
    --disable-cuda-graph-padding
    --disable-decode-cuda-graph
    --disable-piecewise-cuda-graph
    --disable-prefill-cuda-graph
    --enable-breakable-cuda-graph
    --enable-profile-cuda-graph
    --enforce-piecewise-cuda-graph
    --piecewise-cuda-graph-compiler
    --piecewise-cuda-graph-max-tokens
    --piecewise-cuda-graph-tokens

结论: 可以往下跑。


## 3. 写探针脚本（与前几轮同一形状）

In [4]:
import io

P = []
P.append('# -*- coding: utf-8 -*-')
P.append('import argparse, asyncio, json, statistics as st, time')
P.append('import aiohttp')
P.append('URL = "http://127.0.0.1:8000/v1/chat/completions"')
P.append('MODEL = "Qwen/Qwen2.5-0.5B-Instruct"')
P.append('')
P.append('async def one(sess, prompt, max_tokens):')
P.append('    body = {"model": MODEL, "messages": [{"role":"user","content":prompt}],')
P.append('            "max_tokens": max_tokens, "temperature": 0.0, "stream": True}')
P.append('    t0 = time.perf_counter(); ttft=None; n=0; last=t0')
P.append('    async with sess.post(URL, json=body) as resp:')
P.append('        async for raw in resp.content:')
P.append('            line = raw.decode("utf-8").strip()')
P.append('            if not line.startswith("data: ") or line == "data: [DONE]": continue')
P.append('            d = json.loads(line[6:])["choices"][0].get("delta", {})')
P.append('            if d.get("content"):')
P.append('                now = time.perf_counter()')
P.append('                if ttft is None: ttft = now - t0')
P.append('                n += 1; last = now')
P.append('    return dict(ttft=ttft or 0.0, total=last-t0, n_tok=n)')
P.append('')
P.append('async def run_batch(n_conc, n_req, max_tokens):')
P.append('    prompts = ["Explain concept #%d in distributed systems." % i for i in range(n_req)]')
P.append('    sem = asyncio.Semaphore(n_conc)')
P.append('    async def guarded(sess, p):')
P.append('        async with sem: return await one(sess, p, max_tokens)')
P.append('    to = aiohttp.ClientTimeout(total=900)')
P.append('    async with aiohttp.ClientSession(timeout=to) as sess:')
P.append('        await one(sess, "warmup", 4)')
P.append('        t0 = time.perf_counter()')
P.append('        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))')
P.append('        wall = time.perf_counter() - t0')
P.append('    tot = sum(r["n_tok"] for r in rs)')
P.append('    tps = [(r["total"]-r["ttft"])/max(r["n_tok"]-1,1) for r in rs if r["n_tok"]>1]')
P.append('    return dict(conc=n_conc, wall=wall, tput=tot/wall,')
P.append('                ttft_p50=st.median(r["ttft"] for r in rs),')
P.append('                tpot_p50=st.median(tps) if tps else 0.0)')
P.append('')
P.append('async def main(a):')
P.append('    out = []')
P.append('    print("%6s %9s %12s %11s %11s" % ("batch","墙钟s","吞吐tok/s","TTFTp50","TPOTp50"))')
P.append('    print("-"*54)')
P.append('    for c in [int(x) for x in a.conc.split(",")]:')
P.append('        r = await run_batch(c, max(c*4, 16), a.max_tokens)')
P.append('        out.append(r)')
P.append('        print("%6d %9.2f %12.1f %10.1fms %10.2fms" % (')
P.append('              c, r["wall"], r["tput"], r["ttft_p50"]*1e3, r["tpot_p50"]*1e3))')
P.append('    json.dump(out, open(a.out, "w"))')
P.append('')
P.append('if __name__ == "__main__":')
P.append('    ap = argparse.ArgumentParser()')
P.append('    ap.add_argument("--conc", default="4,8,12,16,24,32")')
P.append('    ap.add_argument("--max-tokens", type=int, default=128)')
P.append('    ap.add_argument("--out", default="p.json")')
P.append('    asyncio.run(main(ap.parse_args()))')

io.open("probe.py", "w", encoding="utf-8").write(chr(10).join(P))
print("写出 probe.py，", len(P), "行")


写出 probe.py， 55 行


## 4. 启动器 —— 每次都从日志读回实际捕获的 bs 列表

**不假设参数生效。** 起来之后立刻把 `Capture target decode CUDA graph ... bs=[...]`
这行原文打出来，作为该组配置的事实依据。


In [5]:
import subprocess, sys, time, requests, re

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def serve(extra, tag, wait=600):
    subprocess.run(["pkill", "-f", "sglang.launch_server"], check=False)
    time.sleep(12)
    log = "/content/c_%s.log" % tag
    cmd = [sys.executable, "-m", "sglang.launch_server",
           "--model-path", MODEL, "--host", "127.0.0.1", "--port", "8000",
           "--context-length", "2048"] + extra
    print("启动 [%s]:" % tag, " ".join(cmd[6:]))
    lg = open(log, "w")
    p = subprocess.Popen(cmd, stdout=lg, stderr=subprocess.STDOUT)
    for i in range(wait // 2):
        if p.poll() is not None:
            print("  退出码 %s，日志尾部：" % p.returncode)
            print(open(log).read()[-3000:])
            return None, None
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
                print("  就绪，用时 %ds" % (i * 2))
                txt = open(log).read()
                m = re.search(r"(Capture target decode CUDA graph[^" + chr(10) + r"]*)", txt)
                cap = m.group(1) if m else None
                if cap:
                    print("  捕获原文:", cap[:180])
                else:
                    print("  日志里没有 decode CUDA graph 捕获行（可能已禁用）")
                return p, cap
        except requests.RequestException:
            pass
        time.sleep(2)
    print("  %ds 没起来，日志尾部：" % wait)
    print(open(log).read()[-3000:])
    return None, None


def bs_list(cap):
    if not cap:
        return None
    m = re.search(r"bs=\[([^\]]*)\]", cap)
    return [int(x) for x in m.group(1).split(",")] if m else None


## 5. 三组配置

A 默认（基线复现）／ B 关掉解码 CUDA graph ／ C 把解码图捕获上限提到 32。
三组的 `--mem-fraction-static` 都是 0.80；B、C 与 A 各只差一个参数。


In [6]:
import subprocess, sys, json, os

CONF = [
    ("A_default",   ["--mem-fraction-static", "0.80",
                     "--attention-backend", "triton", "--sampling-backend", "pytorch"]),
    ("B_nograph",   ["--mem-fraction-static", "0.80", "--cuda-graph-backend-decode", "disabled",
                     "--attention-backend", "triton", "--sampling-backend", "pytorch"]),
    ("C_maxbs32",   ["--mem-fraction-static", "0.80", "--cuda-graph-max-bs-decode", "32",
                     "--attention-backend", "triton", "--sampling-backend", "pytorch"]),
]

RES, CAP = {}, {}
for tag, extra in CONF:
    print("=" * 62)
    p, cap = serve(extra, tag)
    CAP[tag] = bs_list(cap)
    print("  解析出的捕获 bs 列表:", CAP[tag])
    if not p:
        print("  起服务失败，跳过", tag)
        continue
    r = subprocess.run([sys.executable, "-u", "probe.py",
                        "--conc", "4,8,12,16,24,32", "--out", "p.json"],
                       capture_output=True, text=True)
    print(r.stdout)
    if os.path.exists("p.json"):
        RES[tag] = {row["conc"]: row for row in json.load(open("p.json"))}
        os.rename("p.json", "res_%s.json" % tag)
print("=" * 62)
print("完成:", list(RES.keys()))


启动 [A_default]: 127.0.0.1 --port 8000 --context-length 2048 --mem-fraction-static 0.80 --attention-backend triton --sampling-backend pytorch
  就绪，用时 232s
  捕获原文: Capture target decode CUDA graph begin. backend=full, num_tokens_per_req=1, bs=[1, 2, 4, 8], avail mem=2.36 GB
  解析出的捕获 bs 列表: [1, 2, 4, 8]
 batch       墙钟s      吞吐tok/s     TTFTp50     TPOTp50
------------------------------------------------------
     4     11.38        180.0       76.2ms       5.70ms
     8      3.57       1147.9       87.8ms       6.29ms
    12     18.17        338.1      118.0ms      34.27ms
    16     18.54        442.0      141.9ms      35.00ms
    24     18.33        670.0      188.4ms      32.80ms
    32     26.92        608.0      241.5ms      35.74ms

启动 [B_nograph]: 127.0.0.1 --port 8000 --context-length 2048 --mem-fraction-static 0.80 --cuda-graph-backend-decode disabled --attention-backend triton --sampling-backend pytorch
  就绪，用时 82s
  日志里没有 decode CUDA graph 捕获行（可能已禁用）
  解析出的捕获 bs 列表: None
 bat

## 6. 判定 —— 台阶跟着捕获范围动了吗

In [7]:
BS = [4, 8, 12, 16, 24, 32]

print("捕获的 bs 列表：")
for tag in ["A_default", "B_nograph", "C_maxbs32"]:
    print("  %-12s %s" % (tag, CAP.get(tag) if CAP.get(tag) else "（无图 / 未取到）"))
print()
print("%-12s" % "TPOT p50 ms", "".join("%9d" % b for b in BS))
print("-" * (12 + 9 * len(BS)))
for tag in ["A_default", "B_nograph", "C_maxbs32"]:
    d = RES.get(tag, {})
    print("%-12s" % tag, "".join(
        ("%9.2f" % (d[b]["tpot_p50"] * 1e3)) if b in d else "%9s" % "-" for b in BS))

def step_at(tag):
    """返回 TPOT 相邻两档跳幅 >2.5 倍的位置（台阶）。"""
    d = RES.get(tag, {})
    out = []
    for i in range(len(BS) - 1):
        a, b = BS[i], BS[i + 1]
        if a in d and b in d and d[a]["tpot_p50"] > 0:
            if d[b]["tpot_p50"] / d[a]["tpot_p50"] > 2.5:
                out.append((a, b))
    return out

print()
for tag in ["A_default", "B_nograph", "C_maxbs32"]:
    print("  %-12s 台阶位置: %s" % (tag, step_at(tag) or "无台阶"))

print()
print("=== 按跑前写死的预测判定 ===")
sa, sb, sc = step_at("A_default"), step_at("B_nograph"), step_at("C_maxbs32")
d8 = RES.get("B_nograph", {}).get(8, {}).get("tpot_p50")
if sa:
    print("A 基线：台阶在", sa, "→ 复现成功")
else:
    print("A 基线：**没有台阶** —— 与前一轮不一致，先解释这个矛盾再往下。")
if d8 is not None:
    print("B 关图：batch 8 的 TPOT = %.2f ms" % (d8 * 1e3))
    if d8 * 1e3 > 15:
        print("   → 关掉图之后 batch 8 也慢了，**台阶由图捕获造成，根因坐实**。")
    else:
        print("   → 关掉图之后 batch 8 仍是 6 ms 量级，**根因解释被推翻**，要重查。")
if CAP.get("C_maxbs32") and sc is not None:
    print("C 提上限：捕获到", CAP["C_maxbs32"], "，台阶位置", sc or "无台阶")
    hi = max(CAP["C_maxbs32"])
    if not sc and hi >= 32:
        print("   → 捕获覆盖到 32，台阶消失，**与预测一致**。")
    elif sc and sc[0][0] >= hi:
        print("   → 台阶移到捕获上限 %d 之后，**与预测一致**。" % hi)
    else:
        print("   → 台阶位置与捕获上限对不上，**预测不成立**，照实记。")


捕获的 bs 列表：
  A_default    [1, 2, 4, 8]
  B_nograph    （无图 / 未取到）
  C_maxbs32    [1, 2, 4, 8, 12, 16, 24, 32]

TPOT p50 ms          4        8       12       16       24       32
------------------------------------------------------------------
A_default         5.70     6.29    34.27    35.00    32.80    35.74
B_nograph        34.17    32.32    33.77    33.91    34.15    36.31
C_maxbs32         5.85     7.88     7.52     7.74     9.71    10.26

  A_default    台阶位置: [(8, 12)]
  B_nograph    台阶位置: 无台阶
  C_maxbs32    台阶位置: 无台阶

=== 按跑前写死的预测判定 ===
A 基线：台阶在 [(8, 12)] → 复现成功
B 关图：batch 8 的 TPOT = 32.32 ms
   → 关掉图之后 batch 8 也慢了，**台阶由图捕获造成，根因坐实**。
C 提上限：捕获到 [1, 2, 4, 8, 12, 16, 24, 32] ，台阶位置 无台阶
   → 捕获覆盖到 32，台阶消失，**与预测一致**。


## 7. 补跑：默认 flashinfer 后端（上一轮未完成）

不传任何 backend 参数。**起不来就是结果**，把失败原因原样打出来。


In [8]:
import subprocess, sys, json, os

p, cap = serve(["--mem-fraction-static", "0.80"], "D_default_backend", wait=600)
FI = None
if p:
    print("  捕获 bs:", bs_list(cap))
    r = subprocess.run([sys.executable, "-u", "probe.py",
                        "--conc", "4,8,12,16,32", "--out", "fi.json"],
                       capture_output=True, text=True)
    print(r.stdout)
    if os.path.exists("fi.json"):
        FI = {row["conc"]: row for row in json.load(open("fi.json"))}
        print("默认后端 TPOT:", {k: round(v["tpot_p50"] * 1e3, 2) for k, v in FI.items()})
else:
    print()
    print("默认后端 600s 内没起来。**这本身是结果**：")
    print("  在 sm_75 上 SGLang 的默认路径不可用，这也解释了为什么前几轮")
    print("  必须显式指定 triton + pytorch。如实记为「默认路径不可用」，")
    print("  但**不要**据此断言凹陷现象在默认路径上也存在或不存在 —— 那仍是未测。")


启动 [D_default_backend]: 127.0.0.1 --port 8000 --context-length 2048 --mem-fraction-static 0.80
  就绪，用时 248s
  捕获原文: Capture target decode CUDA graph begin. backend=full, num_tokens_per_req=1, bs=[1, 2, 4, 8], avail mem=1.91 GB
  捕获 bs: [1, 2, 4, 8]
 batch       墙钟s      吞吐tok/s     TTFTp50     TPOTp50
------------------------------------------------------
     4     24.19         84.7       55.5ms       5.82ms
     8      3.53       1160.7       79.6ms       6.33ms
    12     15.76        389.9      114.3ms      29.92ms
    16     16.84        486.6      122.8ms      32.53ms
    32     17.30        945.6      144.1ms      30.86ms

默认后端 TPOT: {4: 5.82, 8: 6.33, 12: 29.92, 16: 32.53, 32: 30.86}


## 8. 边界

- 单卡 T4、0.5B 模型、2048 上下文、每档单次测量。
- B 组只关**解码**图（`--cuda-graph-backend-decode disabled`），prefill 侧的图设置不动；
  被检验的机制就是解码图，这样反而更干净。
- C 组只改上限（`--cuda-graph-max-bs-decode 32`），显存参数与 A 相同；
  上限能不能真的捕获到 32，以日志读回的 `bs=[...]` 为准。
- 台阶判据用「相邻档 TPOT 跳幅 > 2.5 倍」，是个阈值判断，不是统计检验。
- 若 B 组推翻了根因解释，**前面所有基于 CUDA graph 的表述都要改**，包括已推到
  GitHub 的 README —— 照改，不护着旧结论。
